<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Leo/Random_Forest_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
data_path_profiles = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/linkedin_experience_annotated.csv"
df_profiles = pd.read_csv(data_path_profiles)
df_profiles.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,NaN,ACTIVE,Other,Management,0
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,NaN,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,NaN,ACTIVE,Other,Management,0
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0


In [ ]:
#Fill active Jobs with current date and unknown where we assume that its the active job when job count = 1 (see later code  in 293)
from datetime import date


df_profiles["endDate"] = df_profiles["endDate"].astype(str)

df_profiles.loc[
    ((df_profiles["status"] == "ACTIVE") | (df_profiles["status"] == "UNKNOWN")) & (df_profiles["endDate"].isin(["nan", "NaT"])),
    "endDate"
] = date.today().strftime("%Y-%m")

In [ ]:
#Remove linkedIN  (URL) column

df_profiles.drop(columns=['linkedin'], inplace=True, errors='ignore')
df_profiles.head()

,organization,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,Prokurist,2019-08,2026-01,ACTIVE,Other,Management,0
1,Depot4Design GmbH,CFO,2019-07,2026-01,ACTIVE,Other,Management,0
2,Depot4Design GmbH,Betriebswirtin,2019-07,2026-01,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,Prokuristin,2019-07,2026-01,ACTIVE,Other,Management,0
4,Depot4Design GmbH,CFO,2019-07,2026-01,ACTIVE,Other,Management,0


In [ ]:
#Regularize startDate for job_duration_years
import pandas as pd
import re

def normalize_startdate(val):
    if pd.isna(val):
        return pd.NA

    val = str(val).strip()

    # Case 1: YYYY-MM (bereits korrekt)
    if re.fullmatch(r"\d{4}-\d{2}", val):
        return val

    # Case 2: YYYY \u2192 erg\u00e4nze Januar
    if re.fullmatch(r"\d{4}", val):
        return f"{val}-01"

    # alles andere \u2192 missing
    return pd.NA


df_profiles["startDate"] = df_profiles["startDate"].apply(normalize_startdate)

import pandas as pd
import re

def normalize_enddate(val):
    if pd.isna(val):
        return pd.NA

    val = str(val).strip()

    # Case 1: YYYY-MM (bereits korrekt)
    if re.fullmatch(r"\d{4}-\d{2}", val):
        return val

    # Case 2: YYYY → ergänze Januar
    if re.fullmatch(r"\d{4}", val):
        return f"{val}-01"

    # alles andere → missing
    return pd.NA


df_profiles["endDate"] = df_profiles["endDate"].apply(normalize_enddate)


In [ ]:
# Imputing strategy for Unknown and missing startDate where there is only one (current job)

df_profiles["startDate"] = df_profiles["startDate"].replace(
    ["", "unknown", "novalue", None],
    pd.NA
)

job_counts = df_profiles.groupby("person_id").size()
df_profiles["job_count"] = df_profiles["person_id"].map(job_counts)

# Calculate job_duration_years here before using it
# Use errors='coerce' to handle non-conforming date strings gracefully
start_temp = pd.to_datetime(df_profiles["startDate"], format="%Y-%m", errors='coerce')
end_temp   = pd.to_datetime(df_profiles["endDate"],   format="%Y-%m", errors='coerce')
df_profiles["job_duration_years"] = (end_temp - start_temp).dt.days / 365

#Median Job Duration for imputing
median_duration_years = df_profiles["job_duration_years"].median()
median_duration_months = int(round(median_duration_years * 12))

mask = (
    df_profiles["startDate"].isna() & # Corrected 'df' to 'df_profiles'
    (df_profiles["job_count"] == 1)
)

# endDate als Referenz, sonst heutiges Datum
reference_date = pd.to_datetime(
    df_profiles.loc[mask, "endDate"], # Corrected 'df' to 'df_profiles'
    format="%Y-%m",
    errors="coerce"
).fillna(pd.Timestamp.today())

imputed_start = reference_date - pd.DateOffset(months=median_duration_months)

df_profiles.loc[mask, "startDate"] = imputed_start.dt.strftime("%Y-%m")

In [ ]:
#Drop Rest where theres no start Date and more than one job count (only 23)

df_profiles = df_profiles.drop(
    df_profiles[
        df_profiles["startDate"].isna() &
        (df_profiles["job_count"] > 1)
    ].index
)
#and change status from unknown for imputed to active
df_profiles.loc[
    (df_profiles["status"].str.lower() == "unknown"),
    "status"
] = "ACTIVE"

In [ ]:
#Check for missing values
df_profiles.isna().sum()

,0
organization,0
position,0
startDate,0
endDate,0
status,0
department,0
seniority,0
person_id,0
job_count,0
job_duration_years,0


In [ ]:
#Did we successfully change all Unknown to Active if theres only one job? - Yes, no unknown left
df_profiles["status"].value_counts(dropna=False)

,count
status,
INACTIVE,1897
ACTIVE,718


In [ ]:
#Ready to encode dataframe:

df_profiles.head(75)

,organization,position,startDate,endDate,status,department,seniority,person_id,job_count,job_duration_years
0,Depot4Design GmbH,Prokurist,2019-08,2026-01,ACTIVE,Other,Management,0,6,6.424658
1,Depot4Design GmbH,CFO,2019-07,2026-01,ACTIVE,Other,Management,0,6,6.509589
2,Depot4Design GmbH,Betriebswirtin,2019-07,2026-01,ACTIVE,Other,Professional,0,6,6.509589
3,Depot4Design GmbH,Prokuristin,2019-07,2026-01,ACTIVE,Other,Management,0,6,6.509589
4,Depot4Design GmbH,CFO,2019-07,2026-01,ACTIVE,Other,Management,0,6,6.509589
...,...,...,...,...,...,...,...,...,...,...
70,Lichtenberg School,"Teacher: History, Ethics, French",2008-08,2017-07,INACTIVE,Other,Professional,20,6,8.920548
71,German School Beijing (China),"Teacher: History, Ethics, French",2002-08,2008-07,INACTIVE,Other,Professional,20,6,5.920548
72,"Universities: Chuncheon, Hannam, Hongik (South...",Dr. phil. - German Studies,1990-02,1999-01,INACTIVE,Other,Professional,20,6,8.920548
73,Thurm GmbH,"Geschäftsführer, CMO",2014-07,2026-01,ACTIVE,Marketing,Management,21,3,11.512329


In [ ]:
#Download Code for Team Members
#df_profiles.to_csv("df_profiles.csv", index=False)

#from google.colab import files
#files.download("df_profiles.csv")



Feature Encoding

In [ ]:
df_encoded = df_profiles.copy()

In [ ]:
#Organization encoding - Frequency Encoding

org_freq = df_encoded["organization"].value_counts(normalize=True)
df_encoded["organization_freq"] = df_encoded["organization"].map(org_freq)

df_encoded = df_encoded.drop(columns=["organization"])

In [ ]:
#Position Encoding - Sentence Embeddings

from sentence_transformers import SentenceTransformer
import pandas as pd # Ensure pandas is imported

model = SentenceTransformer("all-MiniLM-L6-v2")
position_embeddings = model.encode(df_encoded["position"].fillna("").tolist())

# Create new column names for the embeddings
embedding_column_names = [f"position_embed_{i}" for i in range(position_embeddings.shape[1])]

# Convert embeddings to a DataFrame and align with df_encoded's index
position_embeddings_df = pd.DataFrame(position_embeddings, index=df_encoded.index, columns=embedding_column_names)

# Drop the original 'position' column and concatenate the new embedding columns
df_encoded = df_encoded.drop(columns=["position"])
df_encoded = pd.concat([df_encoded, position_embeddings_df], axis=1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#startDate & endDate -> Feature Engineering → job_duration_years
start = pd.to_datetime(df_encoded["startDate"], format="%Y-%m", errors='coerce')
end   = pd.to_datetime(df_encoded["endDate"],   format="%Y-%m", errors='coerce')

df_encoded["job_duration_years"] = (end - start).dt.days / 365

df_encoded = df_encoded.drop(columns=["startDate", "endDate"])

In [ ]:
#status → Binary Encoding
df_encoded["status_bin"] = df_encoded["status"].map({"ACTIVE": 1, "INACTIVE": 0})
df_encoded = df_encoded.drop(columns=["status"])


In [ ]:
df_encoded.head()

,department,seniority,person_id,job_count,job_duration_years,organization_freq,position_embed_0,position_embed_1,position_embed_2,position_embed_3,...,position_embed_375,position_embed_376,position_embed_377,position_embed_378,position_embed_379,position_embed_380,position_embed_381,position_embed_382,position_embed_383,status_bin
0,Other,Management,0,6,6.424658,0.001912,-0.038326,0.013078,-0.139571,-0.024655,...,-0.009933,0.025318,0.005560,-0.074910,0.005907,0.137119,-0.000993,0.093262,0.003507,1
1,Other,Management,0,6,6.509589,0.001912,-0.059618,-0.047214,-0.039971,0.075319,...,-0.063512,-0.044259,0.009971,-0.051646,-0.010528,-0.038092,-0.017355,0.083279,0.003473,1
2,Other,Professional,0,6,6.509589,0.001912,-0.048715,0.004592,-0.039356,-0.007562,...,0.019506,0.009547,0.036486,-0.109967,0.016005,-0.005102,-0.030709,0.058480,-0.000570,1
3,Other,Management,0,6,6.509589,0.001912,-0.052659,0.031287,-0.152101,-0.028479,...,0.010195,0.039245,0.024023,-0.046679,0.002175,0.115020,-0.004019,0.089740,0.000141,1
4,Other,Management,0,6,6.509589,0.001912,-0.059618,-0.047214,-0.039971,0.075319,...,-0.063512,-0.044259,0.009971,-0.051646,-0.010528,-0.038092,-0.017355,0.083279,0.003473,1


In [ ]:
df_seniority_raw = df_encoded[embedding_column_names + ["seniority", "department", "person_id"]].copy()


**Datframes for Seniority and Department**

In [ ]:
from sklearn.preprocessing import LabelEncoder

seniority_map = {
    "Junior": 0,
    "Professional": 1,
    "Senior": 2,
    "Lead": 3,
    "Management": 4,
    "Director": 5
}

df_seniority_encoded = df_seniority_raw[
  embedding_column_names + ["seniority", "department", "person_id"]
].copy()

# Target (Ordinal)
df_seniority_encoded["seniority"] = df_seniority_encoded["seniority"].map(seniority_map)


# Department als Feature (Label Encoding)
le_dept = LabelEncoder()
df_seniority_encoded["department_enc"] = le_dept.fit_transform(
    df_seniority_encoded["department"]
)

df_seniority_encoded = df_seniority_encoded.drop(columns=["department"])

In [ ]:
#Department Dataframe for Predicting Department Model
df_department_raw = df_encoded[embedding_column_names +[
    "department",
    "seniority",

]].copy()

df_department_encoded = df_encoded[embedding_column_names +[
    "department",
    "seniority",
]].copy()

# Target (Label Encoding)
le_dep = LabelEncoder()
df_department_encoded["department"] = le_dep.fit_transform(
    df_department_encoded["department"]
)

# Seniority als Feature (Ordinal)
df_department_encoded["seniority_ord"] = df_department_encoded["seniority"].map(seniority_map)
df_department_encoded = df_department_encoded.drop(columns=["seniority"])




Model Predicting Seniority

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = df_seniority_encoded.drop('seniority', axis=1)
y = df_seniority_encoded['seniority']
groups = df_seniority_encoded['person_id']

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=0
)

train_idx, val_idx = next(gss.split(X, y, groups=groups))

train_X = X.iloc[train_idx]
val_X   = X.iloc[val_idx]
train_y = y.iloc[train_idx]
val_y   = y.iloc[val_idx]

In [ ]:
# Random Forest Model

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(train_X, train_y)


RandomForestClassifier(class_weight='balanced', max_depth=12,
                       min_samples_leaf=10, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [ ]:
# Evaluating Model

from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf_model.predict(val_X)

print(classification_report(val_y, y_pred))


              precision    recall  f1-score   support

           0       0.78      0.43      0.56        58
           1       0.69      0.88      0.78       242
           2       0.94      0.73      0.82        45
           3       0.58      0.66      0.62        89
           4       0.96      0.73      0.83       110
           5       0.93      0.61      0.74        44

    accuracy                           0.74       588
   macro avg       0.82      0.67      0.72       588
weighted avg       0.77      0.74      0.74       588



Model Predicting Department

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = df_department_encoded.drop('department', axis=1)
y = df_department_encoded['department']
# Fetch 'person_id' from the original df_encoded DataFrame, aligning by index
groups = df_encoded.loc[df_department_encoded.index, 'person_id']

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=0
)

train_idx, val_idx = next(gss.split(X, y, groups=groups))

train_X = X.iloc[train_idx]
val_X   = X.iloc[val_idx]
train_y = y.iloc[train_idx]
val_y   = y.iloc[val_idx]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model_department = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model_department.fit(train_X, train_y)

RandomForestClassifier(class_weight='balanced', max_depth=12,
                       min_samples_leaf=10, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [ ]:
# Evaluating Model

from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf_model_department.predict(val_X)

print(classification_report(val_y, y_pred))

              precision    recall  f1-score   support

           0       0.67      0.50      0.57        12
           1       0.94      0.65      0.77        23
           2       0.76      0.72      0.74        39
           3       0.80      0.67      0.73        12
           4       0.78      0.54      0.64        13
           5       0.64      0.81      0.72        72
           6       0.48      0.38      0.42        37
           7       0.75      0.82      0.79       284
           8       0.61      0.53      0.57        36
           9       0.75      0.38      0.50        16
          10       0.84      0.70      0.77        44

    accuracy                           0.72       588
   macro avg       0.73      0.61      0.65       588
weighted avg       0.73      0.72      0.72       588



In [ ]:
#Overview Department
le_dep.classes_


array(['Administrative', 'Business Development', 'Consulting',
       'Customer Support', 'Human Resources', 'Information Technology',
       'Marketing', 'Other', 'Project Management', 'Purchasing', 'Sales'],
      dtype=object)

In [ ]:
#Overview -> Problematic Lack of Labels?
df_profiles["department"].value_counts(dropna=False)

,count
department,
Other,1235
Information Technology,309
Sales,219
Consulting,195
Project Management,173
Marketing,133
Administrative,84
Business Development,78
Purchasing,72
